<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/3%20-%20Regress%C3%A3o%20Log%C3%ADstica/3.1_regressao_logistica_iris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 3.1 — Regressão logística (softmax) no Iris: 3 classes

## Objetivos

- Classificar as flores do dataset Iris em 3 espécies a partir de 2 atributos.
- Distinguir logits (scores) de probabilidades e calcular a softmax.
- Treinar o modelo com `nn.CrossEntropyLoss` e `optim.SGD` e acompanhar a fronteira de decisão durante o treinamento.
- Salvar e recuperar os parâmetros treinados (`state_dict`, `torch.save`, `load_state_dict`).
- Avaliar o classificador com acurácia e matriz de confusão.

## O problema

Nos notebooks 2.x a saída do modelo era um número real (o salário). Agora a saída é uma **classe**: dada uma flor, qual é a sua espécie?

- **Entrada**: comprimento da sépala e comprimento da pétala, normalizados para $[0, 1]$.
- **Saída esperada**: o índice da classe correta — 0 (setosa), 1 (versicolor) ou 2 (virginica).
- **Modelo**: uma camada densa linear com 2 entradas e 3 saídas (uma por classe), `nn.Linear(2, 3)`.
- **Função de custo (loss)**: entropia cruzada (cross-entropy). Em alguns materiais aparece o nome "função de perda"; aqui usamos sempre "função de custo".

### Regressão logística binária × regressão softmax

Nos slides, a regressão logística **binária** tem uma única saída, $h_\theta(x) = g(\theta^T x)$, com a sigmoide $g(z) = \dfrac{1}{1 + e^{-z}}$, que estima $P(y = 1 \mid x)$. Com $K > 2$ classes usamos uma saída por classe e trocamos a sigmoide pela **softmax**. Isso é a regressão softmax (ou regressão logística multinomial):

$$z = W x + b, \qquad p_k = \mathrm{softmax}(z)_k = \frac{e^{z_k}}{\sum_{j=0}^{K-1} e^{z_j}}, \qquad k = 0, \dots, K-1$$

- $x$: vetor de entrada (aqui, 2 atributos);
- $W$: matriz de pesos, shape $(K, 2)$; $b$: vetor de bias, shape $(K,)$; aqui $K = 3$;
- $z$: os **logits** (ou scores), um número real por classe, sem limite de faixa;
- $p_k$: a **probabilidade** estimada da classe $k$; as $K$ probabilidades ficam entre 0 e 1 e somam 1.

A função de custo é a cross-entropy média das $M$ amostras de treino:

$$J(W, b) = -\frac{1}{M} \sum_{i=1}^{M} \log p^{(i)}_{y^{(i)}}$$

onde $y^{(i)}$ é o índice da classe correta da amostra $i$ e $p^{(i)}_{y^{(i)}}$ é a probabilidade que o modelo atribuiu a essa classe. Com $K = 2$, a softmax é igual à sigmoide aplicada a $z_1 - z_0$: o caso binário dos slides é um caso particular.

No diagrama abaixo, o número de amostras aparece como $N$ (aqui, $M$). O bloco tracejado roxo mostra que `nn.CrossEntropyLoss` engloba a softmax **e** o custo: ele recebe os logits $z$, não as probabilidades.

![Diagrama: dados de treinamento, rede linear, softmax, custo cross-entropy e gradiente descendente](https://raw.githubusercontent.com/vcasadei/images/master/RegressaoSoftmax_CrossEntropy_Otimizacao.png)

## Importação das bibliotecas

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, Normalize
import pandas as pd
from sklearn.datasets import load_iris

import torch
import torch.nn as nn
import torch.nn.functional as F

from IPython import display

# Semente fixa: os pesos iniciais e os resultados se repetem a cada execução
torch.manual_seed(0)
np.random.seed(0)

Não usamos `torch.autograd.Variable`: ela está obsoleta desde o PyTorch 0.4. Tensores comuns já guardam o histórico de operações quando algum operando tem `requires_grad=True`, como os parâmetros de `nn.Linear`.

## Dados: leitura, normalização e visualização

In [ ]:
iris = load_iris()
X = iris.data[:, ::2]  # colunas 0 e 2: comprimento da sépala e comprimento da pétala (cm)
Y = iris.target        # 0 = setosa, 1 = versicolor, 2 = virginica

# normalização min-max: cada atributo passa a variar entre 0 e 1
Xc = X - X.min(axis=0)
Xs = Xc.max(axis=0)
Xc = Xc / Xs

print('X:', X.shape, ' Y:', Y.shape)
print('classes:', iris.target_names.tolist())

colors = np.array(['r', 'y', 'b'])
for k, name in enumerate(iris.target_names):
    plt.scatter(Xc[Y == k, 0], Xc[Y == k, 1], c=colors[k], edgecolors='k', label=f'{k}: {name}')
plt.xlabel('comprimento da sépala (normalizado)')
plt.ylabel('comprimento da pétala (normalizado)')
plt.legend()
plt.show()

### Preparação dos dados para o PyTorch

A entrada é um tensor `float32` com shape (150, 2). O rótulo é um tensor de inteiros (`long`) com shape (150,), com o **índice** da classe correta: `nn.CrossEntropyLoss` espera esse formato, e não um vetor one-hot.

In [ ]:
X_tensor = torch.tensor(Xc, dtype=torch.float32)  # entrada: float32
Y_tensor = torch.tensor(Y, dtype=torch.long)      # classe: inteiro (long)

print('X_tensor:', X_tensor.shape, X_tensor.dtype)
print('Y_tensor:', Y_tensor.shape, Y_tensor.dtype)

In [ ]:
Y_tensor

## Modelo

A saída da rede tem um neurônio para cada classe. Cada saída $z_k$ é um **logit** (ou score): uma combinação linear das entradas mais o bias, sem nenhuma função aplicada.

![Rede linear com 2 entradas, bias e 3 saídas z0, z1, z2](https://raw.githubusercontent.com/vcasadei/images/master/RegressaoLogisticaRede_2_3.png)

In [ ]:
model = nn.Linear(2, 3)  # 2 entradas e 3 saídas (uma por classe)

`model.state_dict()` devolve um dicionário com os tensores de parâmetros do modelo: `weight` (a matriz $W$) e `bias` (o vetor $b$). É esse dicionário que vamos salvar em disco depois do treinamento.

In [ ]:
model.state_dict()

### Fazendo uma predição

In [ ]:
out = model(X_tensor[:4, :])  # 4 amostras → logits com shape (4, 3)
out

A saída tem `grad_fn` porque foi calculada a partir dos parâmetros, que têm `requires_grad=True`: o PyTorch guarda o histórico para calcular gradientes depois. Os valores são logits: podem ser negativos ou maiores que 1 e não somam 1.

### Calculando as probabilidades com a softmax

- [Wikipedia: Softmax function](https://en.wikipedia.org/wiki/Softmax_function)

![Softmax: scores z0, z1, z2 entram, probabilidades S0, S1, S2 saem e somam 1](https://raw.githubusercontent.com/vcasadei/images/master/softmax.png)

In [ ]:
probs = F.softmax(out, dim=1)  # dim=1: normaliza ao longo das classes (colunas) de cada amostra
print(probs)
print('soma de cada linha:', probs.sum(dim=1))

In [ ]:
F.softmax(out, dim=1) * 100  # em porcentagem

### Predição da classe

In [ ]:
prob, y_pred = torch.max(probs, dim=1)
print(y_pred)
print(prob)

A softmax preserva a ordem dos logits: a classe com o maior logit é também a de maior probabilidade. Para saber só a classe prevista, basta o índice do maior logit.

In [ ]:
print('classe pelo maior logit:        ', torch.argmax(out, dim=1))
print('classe pela maior probabilidade:', torch.argmax(probs, dim=1))

### `nn.CrossEntropyLoss` recebe logits

`nn.CrossEntropyLoss` aplica internamente o log-softmax e calcula $-\log p_{y}$ da classe correta. Por isso ele recebe os **logits** e os índices das classes. A célula abaixo compara o valor calculado pelo PyTorch com a fórmula de $J$ aplicada à mão e mostra, de propósito, o erro comum de aplicar a softmax antes do custo.

In [ ]:
criterion = nn.CrossEntropyLoss()
y4 = Y_tensor[:4]  # classes corretas das 4 amostras

print('CrossEntropyLoss(logits):              ', criterion(out, y4).item())
print('-log(p da classe correta), média à mão:', -torch.log(probs[torch.arange(4), y4]).mean().item())
print('CrossEntropyLoss(softmax) — ERRADO:    ', criterion(probs, y4).item())

Os dois primeiros valores são iguais. O terceiro aplica a softmax duas vezes: o valor é outro e os gradientes ficam menores, o que atrasa o treinamento. A softmax só é aplicada para **interpretar** a saída como probabilidade, nunca antes de `nn.CrossEntropyLoss`.

## Treinamento

### Classe para visualização dinâmica

As funções `predict` e `predict_prob` são usadas para desenhar as fronteiras de decisão. Elas rodam dentro de `torch.no_grad()`: é só inferência, não precisamos montar o grafo de gradientes.

A classe `MyPlotter` redesenha, a cada `every` épocas, três gráficos: o custo ao longo das épocas, as fronteiras de decisão (a classe prevista em cada ponto do plano) e a máxima probabilidade em cada ponto. A animação usa `display.clear_output(wait=True)`, que substitui a figura anterior pela nova.

In [ ]:
def predict(model, inputs):
    # classe prevista: índice do maior logit
    with torch.no_grad():
        outputs = model(inputs)
    _, predicts = torch.max(outputs, 1)
    return predicts.numpy()

def predict_prob(model, inputs):
    # maior probabilidade da softmax em cada amostra
    with torch.no_grad():
        outputs = F.softmax(model(inputs), dim=1)
    predicts, _ = torch.max(outputs, 1)
    return predicts.numpy()

class MyPlotter():
    def __init__(self, model, X, Y, n_epochs, every=250, h=0.01):
        self.model = model
        self.X = X
        self.Y = Y
        self.n_epochs = n_epochs
        self.every = every
        self.loss_hist = []
        x_min, x_max = X[:, 0].min() - .1, X[:, 0].max() + .1
        y_min, y_max = X[:, 1].min() - .1, X[:, 1].max() + .1
        self.xx, self.yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
        self.X_grid = torch.tensor(np.c_[self.xx.ravel(), self.yy.ravel()], dtype=torch.float32)
        self.colors = np.array(['r', 'y', 'b'])
        self.cmap_classes = ListedColormap(['#f6b3b3', '#fff2a8', '#b3cdf6'])  # mesmas cores dos pontos, mais claras

        self.fig, (self.ax1, self.ax2, self.ax3) = plt.subplots(1, 3, figsize=(15, 4.5), dpi=90)
        self.fig.colorbar(plt.cm.ScalarMappable(norm=Normalize(1/3, 1), cmap='magma'),
                          ax=self.ax3, label='máxima probabilidade')
        plt.close(self.fig)  # a figura só aparece quando chamamos display

    def on_epoch_end(self, epoch, loss):
        self.loss_hist.append(loss)
        if epoch % self.every == 0 or epoch == self.n_epochs - 1:
            self.draw(epoch)

    def draw(self, epoch):
        self.ax1.clear()
        self.ax1.plot(np.arange(1, epoch + 2), self.loss_hist, 'b', linewidth=2.0)
        self.ax1.set_xlim(0, self.n_epochs)
        self.ax1.set_xlabel('época')
        self.ax1.set_ylabel('custo (cross-entropy)')
        self.ax1.set_title(f'Custo × época (época {epoch + 1})')
        self.ax1.grid(True)

        Zc = predict(self.model, self.X_grid).reshape(self.xx.shape)
        self.ax2.clear()
        self.ax2.pcolormesh(self.xx, self.yy, Zc, cmap=self.cmap_classes, vmin=0, vmax=2, shading='auto')
        self.ax2.scatter(self.X[:, 0], self.X[:, 1], s=30, c=self.colors[self.Y], edgecolors='k')
        self.ax2.set_xlabel('comprimento da sépala')
        self.ax2.set_ylabel('comprimento da pétala')
        self.ax2.set_title('Fronteiras de decisão')

        Zp = predict_prob(self.model, self.X_grid).reshape(self.xx.shape)
        self.ax3.clear()
        self.ax3.pcolormesh(self.xx, self.yy, Zp, cmap='magma', vmin=1/3, vmax=1, shading='auto')
        self.ax3.scatter(self.X[:, 0], self.X[:, 1], s=30, c=self.colors[self.Y], edgecolors='k')
        self.ax3.set_xlabel('comprimento da sépala')
        self.ax3.set_ylabel('comprimento da pétala')
        self.ax3.set_title('Máxima probabilidade')

        display.clear_output(wait=True)
        display.display(self.fig)

### Laço principal de treinamento da regressão softmax

O otimizador é o mesmo do notebook 2.5: `optim.SGD` faz $w \leftarrow w - \text{lr} \cdot \nabla J$. A função de custo `nn.CrossEntropyLoss` ocupa o lugar do `nn.MSELoss`. A taxa de aprendizado (learning rate) `lr` é um hiperparâmetro: escolhida por nós, não aprendida.

In [ ]:
lr = 0.15

optimizer = torch.optim.SGD(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()  # recebe logits e índices das classes; aplica log-softmax internamente

Cada época (epoch) usa as 150 amostras de uma vez (batch completo): uma época = uma iteração do gradiente descendente. Mini-batches aparecem nos notebooks 3.3 e 3.4. Além do custo, guardamos a acurácia de treino de cada época em `accs`.

In [ ]:
n_epochs = 10000

# Definindo o plotter
my_plot = MyPlotter(model, Xc, Y, n_epochs, every=250)

losses = []
accs = []
for i in range(n_epochs):
    # forward: logits, shape (150, 3) — sem softmax
    output = model(X_tensor)

    # custo: cross-entropy média das 150 amostras
    loss = criterion(output, Y_tensor)

    # retropropagação (backpropagation) e atualização dos parâmetros
    optimizer.zero_grad()  # zera os gradientes acumulados na iteração anterior
    loss.backward()        # calcula dJ/dW e dJ/db
    optimizer.step()       # W ← W − lr·dJ/dW,  b ← b − lr·dJ/db

    # .item() devolve o valor como número Python, sem o histórico de gradientes
    losses.append(loss.item())
    accs.append((output.argmax(dim=1) == Y_tensor).float().mean().item())

    my_plot.on_epoch_end(i, loss.item())

print('Custo final:', losses[-1])

### Custo × acurácia

O custo e a acurácia medem coisas diferentes. A acurácia só conta acertos: ela muda quando alguma amostra troca de lado da fronteira. O custo depende da probabilidade dada à classe correta: continua caindo enquanto essas probabilidades aumentam, mesmo com a acurácia parada.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(losses)
ax1.set_xlabel('época'); ax1.set_ylabel('custo'); ax1.set_title('Custo (cross-entropy)'); ax1.grid(True)
ax2.plot(accs, 'g')
ax2.set_xlabel('época'); ax2.set_ylabel('acurácia'); ax2.set_title('Acurácia de treino'); ax2.grid(True)
plt.tight_layout()
plt.show()

## Avaliação

### Parâmetros após o treinamento

In [ ]:
model.state_dict()

### Salvando os parâmetros treinados

`torch.save(model.state_dict(), caminho)` grava apenas o dicionário de parâmetros, não o objeto `model`. Para usar os parâmetros depois, criamos um modelo com a mesma arquitetura e chamamos `load_state_dict`. O arquivo fica em `./data/models/`.

In [ ]:
os.makedirs('./data/models', exist_ok=True)
model_path = './data/models/RegrLog_Iris.pt'
torch.save(model.state_dict(), model_path)
print(os.listdir('./data/models'))

Para mostrar que a recuperação funciona, zeramos os pesos. `nn.init.constant_` altera o tensor no próprio lugar (o `_` no final indica operação in-place).

In [ ]:
nn.init.constant_(model.weight, 0)
model.state_dict()

### Recuperando os pesos salvos

In [ ]:
param_saved = torch.load(model_path)  # dicionário com os tensores salvos
model.load_state_dict(param_saved)    # copia os valores para os parâmetros do modelo
model.state_dict()

### Exatidão (acurácia) do classificador

A acurácia é o número de acertos dividido pelo número total de amostras. `model.eval()` coloca o modelo em modo de avaliação; para `nn.Linear` não muda nada, mas é o padrão que usaremos sempre que avaliarmos um modelo (faz diferença em camadas como dropout, no módulo 4).

Atenção: aqui a acurácia é medida nas mesmas amostras usadas no treinamento. Ela não diz como o modelo se comporta em flores novas; a separação treino × validação do notebook 2.6 volta no 3.2.

In [ ]:
model.eval()
Y_pred = predict(model, X_tensor)
accuracy = (Y_pred == Y).mean()
print('Acurácia no treino:', accuracy)

### Matriz de confusão

Cada linha é a classe prevista e cada coluna é a classe real. A diagonal conta os acertos.

In [ ]:
cm = pd.crosstab(pd.Series(Y_pred, name='previsto'), pd.Series(Y, name='real'))
cm

## Exercícios

### Perguntas

1. Se a célula de treinamento for executada novamente, ela continua a partir dos parâmetros treinados na última execução. O que é preciso fazer para que o treinamento recomece do início, como na primeira vez?

*Escreva sua resposta aqui.*

2. Refaça o treinamento com diferentes valores de taxa de aprendizado (learning rate), por exemplo 0.01, 0.15, 1.0 e 5.0, partindo sempre dos mesmos pesos iniciais. Compare as curvas de custo.

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

### Exercícios

1. Observando o gráfico de máxima probabilidade, responda e confirme com código:
    1. Qual é a amostra classificada corretamente com maior probabilidade?
    2. Qual é a amostra classificada erradamente com maior probabilidade?
    3. Qual é a amostra classificada corretamente com menor probabilidade?
    4. Qual é a amostra classificada erradamente com menor probabilidade?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

2. Quantos parâmetros possui esta rede? Quantos são pesos da camada densa e quantos são de bias?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

3. Quantas entradas e quantas saídas esta rede possui?

*Escreva sua resposta aqui.*

4. Como se pode calcular a acurácia a partir da matriz de confusão?

In [ ]:
# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

5. O que faz a função `torch.max` quando recebe o argumento `dim`? Quantos valores ela retorna e o que significam?
    - [Documentação de `torch.max`](https://docs.pytorch.org/docs/stable/generated/torch.max.html)

*Escreva sua resposta aqui.*

## Aprendizados

- A regressão softmax gera um logit por classe ($z = Wx + b$); a softmax transforma os logits em probabilidades que somam 1, e a classe prevista é a de maior logit.
- `nn.CrossEntropyLoss` recebe logits e índices das classes e calcula $-\log p$ da classe correta; não se aplica softmax antes dele.
- Com 2 atributos, as fronteiras de decisão de um modelo linear são retas; as amostras com menor probabilidade ficam perto delas.
- Custo e acurácia medem coisas diferentes: o custo continua caindo mesmo quando a acurácia já não muda.
- `state_dict`, `torch.save` e `load_state_dict` salvam e recuperam só os parâmetros; o modelo precisa ser recriado com a mesma arquitetura.
- A mesma dupla softmax + cross-entropy é a camada de saída e o custo do pré-treino de um LLM, que estima a probabilidade do próximo token correto.